# 🦖 Example 6: Deep-Time Paleo-Globes, Modular Colouring & Interactive 3D Preview

This tutorial explains how to create physical, 3D printable globes of ancient Earth across deep geological time (0 to 540 Ma). It covers plate tectonic reconstructions, Scotese PALEOMAP PaleoDEM elevation grids, physics-based full-spectrum filament mixing using 4 base filaments (Brown, Dark Blue, White, Dark Green) for multi-material FDM 3D printing, user-editable discrete banded colormaps with interactive `ipywidgets`, modular direct vertex colouring (`GridColourer`, `SymbolColourer`, `PoleLabelColourer`), and interactive 3D WebGL rendering with `model.preview()`.

---

### 🌎 Key Capabilities Demonstrated:
1. **Paleo Datasets & Catalog**: Querying 108 geological time slices from the Cambrian (540 Ma) to modern day via `globe3d.datasets.paleo`.
2. **Deep-Time Coordinate Tracking**: Tracing modern landmarks (Oxford, UK and Baltimore, USA) across continental drift to the Permo-Triassic supercontinent **Pangaea** (250 Ma) using `PlateModel`.
3. **Full-Spectrum Optical Filament Mixing**: Predicting perceived optical layer mixing colors from physical base spools using a Kubelka-Munk polynomial model (as per *Full Spectrum 3D* / *OrcaSlicer-FullSpectrum* / *PolyMixer*).
4. **In-Notebook Filament Configuration**: Specifying 4 physical base spools (**Brown**, **Dark Blue**, **White**, **Dark Green**) with optical transmission distances (`td`) and generating all available discrete mixtures.
5. **Virtual Palette Swatch Preview**: Displaying bold index-labeled color swatches for direct reference when designing custom hypsometric elevation bands.
6. **User-Editable Hypsometric Colormap (Code & Widget)**: Configuring discrete elevation bands either through editable Python lists or using an interactive `ipywidgets` GUI (`InteractiveColormapBuilder`) with live dual preview.
7. **Modular Vertex Colouring**: Layering colors directly onto 3D vertices using `GridColourer`, `SymbolColourer`, and `PoleLabelColourer` with support for both `modern` (plate-reconstructed) and `paleo` coordinate reference systems.
8. **Accurate Polar Glyph Projection**: Rendering pin-sharp North ('N') and South ('S') labels at the geographic poles via spherical tangent-plane projection, completely free of equirectangular distortion.
9. **Interactive 3D WebGL Preview**: Orbiting, panning, and zooming the textured 3D globe directly inside the notebook using `model.preview()`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import globe3d
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    GridColourer,
    ImageColourer,
    SymbolColourer,
    PoleLabelColourer,
    PlateModel,
    datasets,
    calculate_displacement_scale,
    create_filament_palette,
    create_banded_colormap,
    apply_banded_colormap,
    plot_filament_palette,
    plot_banded_colormap_preview,
    InteractiveColormapBuilder,
)

print(f"globe3d version: {getattr(globe3d, '__version__', '0.1.0')}")

## 1. Inspecting the Scotese PALEOMAP Catalog
`globe3d` includes a registry of 108 reconstructed time slices based on Scotese & Wright (2018) PALEOMAP PaleoDEMs and matching PaleoAtlas textures.

In [ ]:
maps = datasets.paleo.list_scotese_maps()
print(f"Total available geological time slices: {len(maps)}")

print("\nSample time slices:")
for m in maps[::12]:  # Sample every 12th map
    print(f"  {m['age_ma']:>5.1f} Ma: Map {m['map_id']:02d} — {m['name']}")

## 2. Plate Tectonic Reconstructions: Tracing Modern Cities
Using the PALEOMAP Global Plate Model, we can trace where modern cities were situated millions of years ago.
Let's track **Oxford, UK** (51.75°N, -1.26°E) and **Baltimore, USA** (39.29°N, -76.61°W) at the **Permo-Triassic Boundary (250 Ma)**, when all major landmasses were merged into the supercontinent **Pangaea**:

In [ ]:
pm = datasets.paleo.scotese_plate_model()

oxford_modern = [51.7519, -1.2578]
baltimore_modern = [39.2904, -76.6122]

oxford_250ma = pm.reconstruct_points(oxford_modern, age_ma=250.0)[0]
baltimore_250ma = pm.reconstruct_points(baltimore_modern, age_ma=250.0)[0]

print("Oxford, UK:")
print(f"  Modern:         {oxford_modern[0]:.2f}°N, {oxford_modern[1]:.2f}°E")
print(f"  250 Ma Pangaea: {oxford_250ma[0]:.2f}°N, {oxford_250ma[1]:.2f}°E (Subtropical Pangaean interior)\n")

print("Baltimore, USA:")
print(f"  Modern:         {baltimore_modern[0]:.2f}°N, {baltimore_modern[1]:.2f}°E")
print(f"  250 Ma Pangaea: {baltimore_250ma[0]:.2f}°N, {baltimore_250ma[1]:.2f}°E (Central Pangaean suture)")

## 3. Full-Spectrum Optical Filament Mixing for Multi-Material 3D Printing

In multi-material FDM 3D printing (e.g. Bambu AMS, Prusa XL / MMU, or tool-changers), alternating ultra-thin layers (0.08–0.12 mm) of semi-translucent base filaments causes light to pass through adjacent layers, producing a perceived blend of color (optical mixing / pointillism).

`globe3d.filaments` includes a learned degree-4 Kubelka-Munk polynomial model (identical to *Full Spectrum 3D* / *PolyMixer*) that accurately predicts perceived layered blends from base filament colors and transmission distances (`td`).

## 4. In-Notebook Filament Configuration & Available Color Mixing

Here we specify our 4 base physical filament spools loaded in the 3D printer:
- **Brown**: For uplands, plateaus, and mountain ranges.
- **Dark Blue**: For deep oceanic abyssal plains.
- **White**: High transmission distance (`td=5.0`), creates shallow shelf tints and snow-capped alpine peaks.
- **Dark Green**: For coastal plains and continental lowlands.

The cell below configures the spools and computes all available discrete 4-layer optical mixtures, displaying an index table and a swatch palette labeled by **integer index** for easy selection:

In [ ]:
# 1. Define physical base filament spools with transmission distances (td in mm)
filaments_4 = [
    {"name": "Brown", "color": [90, 50, 30], "td": 2.5},
    {"name": "Dark Blue", "color": [20, 40, 95], "td": 3.0},
    {"name": "White", "color": [245, 245, 245], "td": 5.0},
    {"name": "Dark Green", "color": [30, 85, 45], "td": 2.8},
]

print("Base Filament Spools (Loaded in AMS):")
for idx, f in enumerate(filaments_4):
    hex_code = f"#{f['color'][0]:02x}{f['color'][1]:02x}{f['color'][2]:02x}"
    print(f"  [{idx}] {f['name']:<12} RGB: {str(f['color']):<16} Hex: {hex_code}  td: {f['td']} mm")

# 2. Compute all available discrete 4-layer color mixtures across the 4 spools
palette_4 = create_filament_palette(filaments_4, ratios=[0.25, 0.5, 0.75], include_tints=False)
print(f"\nGenerated {len(palette_4)} printable optical color mixtures across the 4 base spools!")

# 3. Print indexed reference table for easy selection
print("\nAvailable Palette Colors (Reference Index -> Name & Hex):")
print("-" * 75)
for i, p in enumerate(palette_4):
    hex_c = p.get('hex', f"#{p['color'][0]:02x}{p['color'][1]:02x}{p['color'][2]:02x}")
    recipe = p.get('layers', p['name'])
    print(f"  Index {i:>2}: {p['name']:<30} Hex: {hex_c}  Layers: {recipe}")

# 4. Display swatch preview with integer index numbers clearly visible
fig, ax = plt.subplots(figsize=(18, 4))
plot_filament_palette(palette_4, title="4-Filament Virtual Palette (Index Labeled for Selection)", max_swatches=len(palette_4), ax=ax)
plt.show()

## 5. Loading the 250 Ma PaleoDEM Elevation Grid

We load the Permo-Triassic (250 Ma) Scotese PaleoDEM elevation grid. This grid provides the topographical data for our discrete hypsometric colormaps, 2D texture rendering, and 3D mesh displacement:

In [ ]:
# Load 250 Ma PaleoDEM elevation grid and texture from datasets
dem_grid, texture_path, meta = datasets.paleo.scotese(250.0)
actual_age = meta["age_ma"]
raw_texture = Image.open(texture_path)

print(f"Loaded: {meta['name']} ({actual_age} Ma)")
print(f"PaleoDEM Grid shape: {dem_grid.data.shape}, range: [{dem_grid.data.min():.0f}m, {dem_grid.data.max():.0f}m]")

## 6. User-Editable Colormap Builder (Code Version)

In this cell, you can manually configure your discrete elevation bands:
- Reference any color from `palette_4` by its integer **`index`** (from the virtual palette above).
- Set the upper elevation boundary (`threshold` in meters).
- Choose rich, distinct filament mixtures for oceanic abyssal plains, continental slopes, shallow shelves, lowlands, highlands, and peaks.

The cell dynamically builds the matplotlib colormap and norm, displaying a uniformly spaced colorbar preview where all bands remain distinct and visible:

In [ ]:
# User-editable elevation band definitions:
# Edit the palette index, elevation threshold (meters), and descriptive label:
elevation_bands = [
    {"index": 1,  "threshold": -3500, "name": "Abyssal Panthalassa (Dark Blue)"},
    {"index": 27, "threshold": -200,  "name": "Continental Slope (Steel Teal)"},
    {"index": 13, "threshold": 0,     "name": "Continental Shelf & Shallow Seas (Cyan)"},
    {"index": 3,  "threshold": 300,   "name": "Lowland Plains (Dark Green)"},
    {"index": 32, "threshold": 1000,  "name": "Savannas & Plateaus (Olive Gold)"},
    {"index": 6,  "threshold": 2200,  "name": "Highlands & Midlands (Warm Ochre)"},
    {"index": 0,  "threshold": 4000,  "name": "Mountain Ranges (Brown)"},
    {"index": 2,  "threshold": None,  "name": "Glaciers & Alpine Snow (White)"},
]

# Extract selected colors and boundaries
selected_colors = [palette_4[b["index"]]["color"] for b in elevation_bands]
thresholds = [b["threshold"] for b in elevation_bands if b["threshold"] is not None]
band_names = [b["name"] for b in elevation_bands]

# Build discrete banded colormap
cmap, norm = create_banded_colormap(selected_colors, thresholds)

print("Configured Discrete Elevation Colormap:")
print("-" * 85)
prev = "-Inf"
for b in elevation_bands:
    thresh_str = f"{prev} to {b['threshold']} m" if b['threshold'] is not None else f"> {prev} m"
    hex_c = f"#{palette_4[b['index']]['color'][0]:02x}{palette_4[b['index']]['color'][1]:02x}{palette_4[b['index']]['color'][2]:02x}"
    print(f"  Palette #{b['index']:<2} | {b['name']:<42} | {thresh_str:<22} | Hex: {hex_c}")
    prev = f"{b['threshold']}"

# Preview the discrete banded colorbar (with uniform spacing for clear visual distinction)
fig, ax = plt.subplots(figsize=(13, 2.0))
plot_banded_colormap_preview(cmap, norm, thresholds, band_labels=band_names, title="User-Defined Elevation Bands", ax=ax)
plt.show()

### 6b. Alternative: Interactive `ipywidgets` Colormap Builder

As an interactive alternative to editing the code list above, `InteractiveColormapBuilder` provides a live GUI:
- **Color Dropdowns**: Select any filament mixture from `palette_4` with live swatch indicators.
- **Threshold Boxes**: Adjust upper elevation boundaries (in meters) interactively.
- **Live Dual Preview**: Updates both the discrete colorbar and the 2D equirectangular terrain map in real-time as you tweak values!

In [ ]:
# Launch interactive widget colormap builder with live dual preview
builder = InteractiveColormapBuilder(palette_4, dem_grid=dem_grid, default_bands=elevation_bands)
builder.show()

# Tip: To adopt edits made in the interactive widget into the downstream 3D pipeline, simply run:
# cmap, norm, selected_colors, thresholds, band_names = builder.get_colormap()

## 7. 2D Texture Preview (North Correctly Oriented)

We apply our discrete banded colormap to `dem_grid` via `apply_banded_colormap(dem_grid, ...)`.
Notice that the latitude axis is automatically oriented with **North at the top**, matching standard equirectangular projections and aligning perfectly with the raw Scotese PaleoAtlas map:

In [ ]:
# Apply banded colormap to dem_grid (automatically flips Y axis so North is at the top)
banded_texture_arr = apply_banded_colormap(dem_grid, selected_colors, thresholds)
banded_img = Image.fromarray(banded_texture_arr)

# 2D Comparison Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(raw_texture)
axes[0].set_title("1. Raw Scotese PaleoAtlas (250 Ma) — North at Top", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(banded_img)
axes[1].set_title("2. Banded Texture Preview (4 Base Filaments) — North at Top", fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 8. Modular 3D Globe Construction with Granular Colourers

In `globe3d`, coloring is completely modular and layered:
- **`GlobeModel`**: Instantiates the un-displaced spherical mesh (with hollowing).
- **`GridDisplacer`**: Displaces vertices radially using the PaleoDEM grid and applies a vertical shelf step (`coastline_step=1000.0` m) to accentuate shorelines.
- **`GridColourer`**: Evaluates vertex elevations against our discrete banded colormap and assigns exact band colors directly to vertices.
- **`SymbolColourer`**: Projects geometric symbols (stars, pins, circles, etc.) onto vertices at specified coordinates. Supports `reference="modern"` (automatically reconstructed via `plate_model` and `age`) or `reference="paleo"`, with chosen colors and radii.
- **`PoleLabelColourer`**: Projects sharp, undistorted North ('N') and South ('S') glyphs directly onto polar vertices via spherical tangent-plane projection, completely avoiding equirectangular pole stretching.

Each colourer layers non-destructively over existing vertex colors:

In [ ]:
# 1. Create base un-displaced sphere mesh
model = GlobeModel(
    method="fibonacci",
    n_points=25000,
    radius=100.0,
    hollow=True,
    inner_ratio=0.9,
)

# 2. Apply topographic displacement from PaleoDEM
scale = calculate_displacement_scale(100.0, grid_units="m") * 35.0
model.outer.displace(GridDisplacer(dem_grid, coastline_step=1000.0), scale=scale)

# 3. Base hypsometric coloring directly on vertices using our editable banded colormap
model.outer.colour(GridColourer(dem_grid, colormap=cmap, norm=norm))

# 4. Modular SymbolColourers for modern landmarks reconstructed to 250 Ma Pangaea
model.outer.colour(
    SymbolColourer(
        symbol="star",
        lat=51.7519, lon=-1.2578, # Oxford, UK
        reference="modern",
        age=actual_age,
        plate_model=pm,
        color="red",
        radius_degrees=3.0,
    )
)
model.outer.colour(
    SymbolColourer(
        symbol="star",
        lat=39.2904, lon=-76.6122, # Baltimore, USA
        reference="modern",
        age=actual_age,
        plate_model=pm,
        color="yellow",
        radius_degrees=3.0,
    )
)

# 5. Modular PoleLabelColourer for North ('N') and South ('S') polar markers
model.outer.colour(
    PoleLabelColourer(
        radius_degrees=3.5,
        north_color="red",
        south_color="cyan",
    )
)

print("Built modular 250 Ma Pangaea GlobeModel with direct vertex coloring!")
print(f"Outer mesh vertices: {len(model.outer.vertices):,}")
print(f"Inner mesh vertices: {len(model.inner.vertices):,}")
print(f"Radial span: [{np.linalg.norm(model.outer.vertices, axis=1).min():.1f} mm, {np.linalg.norm(model.outer.vertices, axis=1).max():.1f} mm]")

## 9. Interactive 3D WebGL Mesh Preview with `model.preview()`

Calling `model.preview()` embeds an interactive 3D WebGL Three.js viewer directly in the notebook:
- **Orbit / Rotate**: Click and drag with the left mouse button to rotate around Pangaea and inspect the ancient globe from any angle.
- **Pan**: Right-click (or Shift + click) and drag to pan the camera.
- **Zoom**: Scroll wheel to inspect local topography, shelf steps, reconstructed city stars, and polar 'N' / 'S' markers.

Notice how the North Pole 'N' label is now rendered crisp and undistorted directly on the polar vertices:

In [ ]:
# Launch interactive 3D WebGL preview of the whole globe
model.preview()

You can also preview individual hemispheres (e.g. Northern hemisphere capped at the equator) using `part='upper'` to inspect the hollowed interior and wall thickness:

In [ ]:
# Preview the Northern hemisphere
model.preview(part="upper")

## 10. Magnet Alignment Voids & 3D Print Export

For printing on a standard desktop 3D printer:
1. `configure_magnets()` inserts recessed cylindrical pockets along the equator with alignment bosses.
2. `export_hemispheres()` splits the model along the equator into Northern and Southern hemispheres (with flat equatorial bases for bed adhesion).

In [ ]:
# Configure magnet cavities (e.g. 4 neodymium magnets of 6x3mm along the equator)
# model.configure_magnets(magnet_radius=3.1, magnet_thickness=3.2, count=4)

# Export hemispheres ready for slicer import
# model.export_hemispheres(
#     upper_path="paleo_250ma_north.obj",
#     lower_path="paleo_250ma_south.obj",
#     flip_lower=True,
# )

print("Paleo-globe pipeline complete and ready for physical printing!")